# Paper 2 — Adjusted Multivariable Analysis of Detector-Indicated AI Rate

This notebook performs a cautious adjusted analysis for **Paper 2: AI-Writing Signals in Saudi University Assignments**.

It is designed to work in Google Colab with the project Google Drive folders, but it also works locally if the dataset is placed beside the notebook.

**Main outcome:** Combined AI rate, kept on the original 0–100 scale and converted to a 0–1 fractional outcome.

**Main predictors:** assignment type, language, word count, major, university, and year grouping.

**Caution:** This is a corpus-level, observational/document-based analysis of detector-indicated signals. It must not be interpreted causally or as proof that any single assignment was AI-generated.

In [ ]:
# Setup: imports, environment detection, and output directory preparation
import re
import sys
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from IPython.display import display

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2_contingency

# Suppress specific noisy warnings from third-party libraries
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Display settings for pandas outputs
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

# Optional: enable DOCX export if python-docx is installed
try:
    from docx import Document

    DOCX_AVAILABLE = True
except Exception:
    DOCX_AVAILABLE = False

# Detect execution in Google Colab and mount drive if present
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

# Output directory: use PROJECT_DATA_PATH in Colab, otherwise local folder
OUTPUT_DIR = (
    Path("${PROJECT_DATA_PATH}")
    if IN_COLAB
    else Path("Paper2_Adjusted_Multivariable_Analysis_Outputs")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR.resolve())

## 1. Load the cleaned assignment-level dataset

The earlier project notebooks used Google Drive paths under `AI_RSRCH_FILES/Assignments Part/...`. The combined primary-sample output is expected to be `AI_Detection_Results_Combined.xlsx` or a corresponding CSV file under a `CSV Results` folder.

If no candidate path is found, the notebook will ask for manual upload in Colab.

In [ ]:
# =========================================================
# Locate data file from likely locations (project variables, common filenames)
# =========================================================
PATH_CANDIDATES = [
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("${PROJECT_DATA_PATH}"),
    Path("/content/AI_Detection_Results_Combined.xlsx"),
    Path("/content/AI_Detection_Results_Combined.csv"),
    Path("AI_Detection_Results_Combined.xlsx"),
    Path("AI_Detection_Results_Combined.csv"),
    Path("/mnt/data/paper2_work/CSV Results/AI_Detection_Results_Combined.xlsx"),
]


def find_first_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None


DATA_PATH = find_first_existing_path(PATH_CANDIDATES)

# If running in Colab and no candidate found, prompt user to upload the file
if DATA_PATH is None and IN_COLAB:
    from google.colab import files

    print(
        "No data file found in path candidates. Please upload AI_Detection_Results_Combined.xlsx or CSV."
    )
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = Path(next(iter(uploaded.keys())))

# Fail-fast if still not found; user must provide one of the expected filenames
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find AI_Detection_Results_Combined.xlsx/csv. Edit PATH_CANDIDATES or upload/place the file beside this notebook."
    )

print("Loading:", DATA_PATH)
# Read Excel or CSV depending on file extension
if DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    raw = pd.read_excel(DATA_PATH)
else:
    raw = pd.read_csv(DATA_PATH)

print("Raw shape:", raw.shape)
display(raw.head())
print(raw.columns.tolist())

In [ ]:
# =========================================================
# 2. Standardize column names and validate expected variables
# =========================================================
def normalize_colname(c):
    c = str(c).strip().lower()
    c = re.sub(r"[%()]", "", c)
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c


raw_to_norm = {c: normalize_colname(c) for c in raw.columns}
df = raw.rename(columns=raw_to_norm).copy()

aliases = {
    "language": "language",
    "university": "university",
    "major": "major",
    "assignment_type": "assignment_type",
    "assignmenttype": "assignment_type",
    "year": "year_original",
    "year_group": "year_original",
    "year_grouping": "year_original",
    "file_name": "file_name",
    "filename": "file_name",
    "word_count": "word_count",
    "weighted_ai_rate": "combined_ai_rate_100",
    "combined_ai_rate": "combined_ai_rate_100",
    "combined_ai_rate_percent": "combined_ai_rate_100",
    "combined_ai_score": "combined_ai_rate_100",
    "gptzero_ai_rate_percent": "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent": "pangram_ai_rate_percent",
    "sapling_ai_rate_percent": "sapling_ai_rate_percent",
    "isgen_ai_rate_percent": "isgen_ai_rate_percent",
    "weighted_result": "combined_result",
}
df = df.rename(columns={c: aliases[c] for c in df.columns if c in aliases})

required = [
    "combined_ai_rate_100",
    "language",
    "assignment_type",
    "word_count",
    "major",
    "university",
    "year_original",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns after standardization: {missing}\nAvailable columns: {df.columns.tolist()}"
    )

keep_cols = [
    c
    for c in [
        "file_name",
        "language",
        "university",
        "major",
        "assignment_type",
        "year_original",
        "word_count",
        "gptzero_ai_rate_percent",
        "pangram_ai_rate_percent",
        "sapling_ai_rate_percent",
        "isgen_ai_rate_percent",
        "combined_ai_rate_100",
        "combined_result",
    ]
    if c in df.columns
]
df = df[keep_cols].copy()

for col in ["language", "university", "major", "assignment_type", "year_original"]:
    df[col] = df[col].astype(str).str.strip()

df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce")
df["combined_ai_rate_100"] = pd.to_numeric(df["combined_ai_rate_100"], errors="coerce")
for c in [
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(
    subset=[
        "combined_ai_rate_100",
        "word_count",
        "language",
        "assignment_type",
        "major",
        "university",
        "year_original",
    ]
).copy()
df["combined_ai_rate_100"] = df["combined_ai_rate_100"].clip(0, 100)
df["combined_ai_rate_prop"] = df["combined_ai_rate_100"] / 100.0
df["log_word_count"] = np.log1p(df["word_count"])


def collapse_year_row(row):
    """Collapse year labels and correct the known Taif/Medical parsing issue.

    In the original combined sheet, some Taif medical rows have Year == "Medical".
    This came from parsing paths like Taif/Medical/First Year/... by taking the
    second path segment as the year. Because Medical is the major/program label,
    not a year, we recover the year from assignment type for these rows:
    OSCE and Manuscripts = First Year; Research Projects = Second Year.
    """
    x = str(row["year_original"]).strip()
    major = str(row.get("major", "")).strip()
    atype = str(row.get("assignment_type", "")).strip()

    if x == "Medical" and major == "Med":
        if atype in {"OSCE", "Manuscripts"}:
            return "First Year"
        if atype == "Research Projects":
            return "Second Year"
        return np.nan
    if "First Year" in x:
        return "First Year"
    if "Second Year" in x:
        return "Second Year"
    return x


df["year_grouping"] = df.apply(collapse_year_row, axis=1)

bad_years = sorted(set(df["year_grouping"].dropna()) - {"First Year", "Second Year"})
if bad_years:
    print("Warning: unexpected year_grouping categories remain:", bad_years)


def make_assignment_context(row):
    language = str(row["language"])
    atype = str(row["assignment_type"])
    major = str(row["major"])
    if language == "Arabic" and atype == "Article":
        return "Arabic Article / General"
    if atype == "Project Report" and major == "MechEng":
        return "Engineering Project Report"
    return atype


df["assignment_context"] = df.apply(make_assignment_context, axis=1)

print("Clean analytic shape:", df.shape)
print(
    "Outcome range:", df["combined_ai_rate_100"].min(), df["combined_ai_rate_100"].max()
)
display(df.head())

## 2. Descriptive cross-tabs: overlap among grouping variables

These tables identify whether language, assignment type, major, university, and year grouping can be separated statistically. Strong overlap means adjusted coefficients should be interpreted cautiously.

In [ ]:
# =========================================================
# 3. Descriptive summaries and crosstabs
# =========================================================
cat_vars = [
    "language",
    "assignment_type",
    "assignment_context",
    "major",
    "university",
    "year_grouping",
]

category_counts = []
for col in cat_vars:
    vc = df[col].value_counts(dropna=False).reset_index()
    vc.columns = ["category", "n"]
    vc.insert(0, "variable", col)
    vc["percent"] = 100 * vc["n"] / len(df)
    category_counts.append(vc)
category_counts = pd.concat(category_counts, ignore_index=True)
print("Category counts")
display(category_counts)

crosstab_dict = {}
for a, b in combinations(
    ["language", "assignment_type", "major", "university", "year_grouping"], 2
):
    ct = pd.crosstab(df[a], df[b])
    crosstab_dict[f"{a}_by_{b}"] = ct
    print(f"\nCrosstab: {a} by {b}")
    display(ct)

In [ ]:
# =========================================================
# 4. Association and overlap diagnostics for categorical variables
# =========================================================
def cramers_v(confusion_matrix):
    chi2, p, dof, expected = chi2_contingency(confusion_matrix)
    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan, p
    r, k = confusion_matrix.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1)) if n > 1 else 0
    rcorr = r - ((r - 1) ** 2) / (n - 1) if n > 1 else r
    kcorr = k - ((k - 1) ** 2) / (n - 1) if n > 1 else k
    denom = min((kcorr - 1), (rcorr - 1))
    v = np.sqrt(phi2corr / denom) if denom > 0 else np.nan
    return v, p


def overlap_metrics(ct):
    row_props = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0)
    col_props = ct.div(ct.sum(axis=0).replace(0, np.nan), axis=1)
    return {
        "max_row_purity": float(row_props.max(axis=1).max()),
        "max_col_purity": float(col_props.max(axis=0).max()),
        "cells_n_lt_5": int((ct < 5).sum().sum()),
        "cells_n_0": int((ct == 0).sum().sum()),
        "n_rows": ct.shape[0],
        "n_cols": ct.shape[1],
    }


assoc_rows = []
for a, b in combinations(
    ["language", "assignment_type", "major", "university", "year_grouping"], 2
):
    ct = pd.crosstab(df[a], df[b])
    v, p = cramers_v(ct)
    met = overlap_metrics(ct)
    assoc_rows.append(
        {
            "var1": a,
            "var2": b,
            "cramers_v": v,
            "chi_square_p": p,
            **met,
            "near_perfect_overlap_flag": (
                met["max_row_purity"] >= 0.95
                or met["max_col_purity"] >= 0.95
                or met["cells_n_0"] > 0
            ),
        }
    )
assoc = pd.DataFrame(assoc_rows).sort_values(
    ["cramers_v", "max_row_purity"], ascending=False
)
print("Categorical association / overlap diagnostics")
display(assoc)

sparse_threshold = 20
sparse_rows = []
for col in cat_vars:
    vc = df[col].value_counts(dropna=False)
    for level, n in vc.items():
        if n < sparse_threshold:
            sparse_rows.append(
                {
                    "variable": col,
                    "category": level,
                    "n": int(n),
                    "sparse_threshold": sparse_threshold,
                }
            )
sparse_categories = pd.DataFrame(sparse_rows)
print("Sparse categories (<20 files)")
display(
    sparse_categories
    if len(sparse_categories)
    else pd.DataFrame({"message": ["No sparse categories below threshold."]})
)

In [ ]:
# =========================================================
# Compute variance inflation factors (VIF) after one-hot encoding
# =========================================================

def compute_vif(data, categorical_predictors, numeric_predictors):
    # Build design matrix from numeric columns and one-hot encoded categorical columns
    X_parts = []
    if numeric_predictors:
        X_parts.append(data[numeric_predictors].astype(float))
    for col in categorical_predictors:
        # Create dummy variables; drop_first avoids perfect multicollinearity from categorical set
        dummies = pd.get_dummies(
            data[col].astype(str), prefix=col, drop_first=True, dtype=float
        )
        X_parts.append(dummies)
    X = pd.concat(X_parts, axis=1)

    # Remove constant columns (zero variance) and rows with infinite or missing values
    X = X.loc[:, X.nunique(dropna=False) > 1]
    X = X.replace([np.inf, -np.inf], np.nan).dropna()

    # Add intercept for VIF calculation; will be skipped in the loop
    X_const = sm.add_constant(X, has_constant="add")
    rows = []
    for i, col in enumerate(X_const.columns):
        if col == "const":
            # Skip intercept column when reporting VIF
            continue
        try:
            vif = variance_inflation_factor(X_const.values, i)
        except Exception:
            # Treat any failure in computation as extremely high multicollinearity
            vif = np.inf
        rows.append({"term": col, "VIF": vif})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False), X


full_cat_predictors = [
    "language",
    "assignment_type",
    "major",
    "university",
    "year_grouping",
]
vif_full, X_full = compute_vif(df, full_cat_predictors, ["log_word_count"])
print("Full-model VIF after one-hot encoding")
display(vif_full.head(50))

vif_context, X_context = compute_vif(df, ["assignment_context"], ["log_word_count"])
print("Reduced context-model VIF")
display(vif_context)

## 3. Modeling utilities

Models are deliberately cautious:

1. **Robust OLS with HC3 errors** on the 0–100 Combined AI rate.
2. **Fractional logit / quasi-binomial-style GLM** on the 0–1 Combined AI rate.
3. **Sensitivity models excluding code files.**
4. **Reduced context models** that avoid simultaneously adjusting for highly overlapping grouping variables.

The full model is useful as a sensitivity check, but if diagnostics show near-perfect overlap, the reduced assignment-context model should receive greater interpretive weight.

In [ ]:
# =========================================================
# 6. Modeling helpers
# =========================================================
def fit_ols_hc3(formula, data, model_name):
    res = smf.ols(formula, data=data).fit(cov_type="HC3")
    res.model_name = model_name
    res.model_family = "OLS-HC3"
    return res


def fit_frac_logit(formula, data, model_name):
    try:
        res = smf.glm(formula, data=data, family=sm.families.Binomial()).fit(
            cov_type="HC3"
        )
    except Exception:
        res = smf.glm(formula, data=data, family=sm.families.Binomial()).fit(
            cov_type="HC0"
        )
    res.model_name = model_name
    res.model_family = "Fractional logit GLM"
    return res


def pseudo_r2_mcfadden(res):
    try:
        return 1 - (res.llf / res.llnull)
    except Exception:
        return np.nan


def model_fit_row(res):
    out = {
        "model": getattr(res, "model_name", "model"),
        "family": getattr(res, "model_family", type(res).__name__),
        "n": int(res.nobs),
    }
    if hasattr(res, "rsquared_adj"):
        out["adjusted_R2"] = float(res.rsquared_adj)
        out["R2"] = float(res.rsquared)
        out["AIC"] = float(res.aic)
        out["BIC"] = float(res.bic) if hasattr(res, "bic") else np.nan
    else:
        out["pseudo_R2_McFadden"] = pseudo_r2_mcfadden(res)
        out["AIC"] = float(res.aic)
        out["BIC"] = float(res.bic) if hasattr(res, "bic") else np.nan
    return out


def tidy_model(res):
    params = res.params
    conf = res.conf_int()
    pvals = res.pvalues
    bse = res.bse
    stat = params / bse
    rows = []
    df_resid = getattr(res, "df_resid", np.nan)
    is_ols = hasattr(res, "rsquared_adj")
    for term in params.index:
        if term == "Intercept":
            continue
        est = params.loc[term]
        lo, hi = conf.loc[term].tolist()
        p = pvals.loc[term]
        zt = stat.loc[term]
        partial_eta2 = np.nan
        if is_ols and not np.isnan(df_resid):
            partial_eta2 = (zt**2) / (zt**2 + df_resid)
        rows.append(
            {
                "model": getattr(res, "model_name", "model"),
                "family": getattr(res, "model_family", type(res).__name__),
                "term": term,
                "estimate": est,
                "std_error": bse.loc[term],
                "CI_low": lo,
                "CI_high": hi,
                "statistic_t_or_z": zt,
                "p": p,
                "partial_eta2_for_OLS_coeff": partial_eta2,
            }
        )
    return pd.DataFrame(rows)


def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< .001"
    return f"{p:.3f}".replace("0.", ".")


def apa_table(df_in, estimate_digits=2):
    df2 = df_in.copy()
    for c in [
        "estimate",
        "std_error",
        "CI_low",
        "CI_high",
        "statistic_t_or_z",
        "partial_eta2_for_OLS_coeff",
    ]:
        if c in df2.columns:
            df2[c] = df2[c].map(
                lambda x: "" if pd.isna(x) else f"{x:.{estimate_digits}f}"
            )
    if "p" in df2.columns:
        df2["p"] = df2["p"].map(format_p)
    return df2

In [ ]:
# =========================================================
# 7. Fit full, sensitivity, and reduced models
# =========================================================
full_formula_ols = "combined_ai_rate_100 ~ log_word_count + C(language) + C(assignment_type) + C(major) + C(university) + C(year_grouping)"
full_formula_glm = "combined_ai_rate_prop ~ log_word_count + C(language) + C(assignment_type) + C(major) + C(university) + C(year_grouping)"
context_formula_ols = "combined_ai_rate_100 ~ log_word_count + C(assignment_context)"
context_formula_glm = "combined_ai_rate_prop ~ log_word_count + C(assignment_context)"
language_formula_ols = "combined_ai_rate_100 ~ log_word_count + C(language)"
language_formula_glm = "combined_ai_rate_prop ~ log_word_count + C(language)"

df_text = df[df["language"].str.lower() != "code"].copy()
models = []
model_errors = []


def try_fit(label, fit_func, formula, data):
    try:
        res = fit_func(formula, data, label)
        models.append(res)
        print(f"Fitted: {label} | n={int(res.nobs)}")
        return res
    except Exception as e:
        model_errors.append({"model": label, "formula": formula, "error": repr(e)})
        print(f"FAILED: {label}\n  {e}")
        return None


ols_full = try_fit("Full adjusted OLS-HC3", fit_ols_hc3, full_formula_ols, df)
glm_full = try_fit(
    "Full adjusted fractional logit", fit_frac_logit, full_formula_glm, df
)
ols_context = try_fit(
    "Reduced assignment-context OLS-HC3", fit_ols_hc3, context_formula_ols, df
)
glm_context = try_fit(
    "Reduced assignment-context fractional logit",
    fit_frac_logit,
    context_formula_glm,
    df,
)
ols_language = try_fit(
    "Reduced language-only OLS-HC3", fit_ols_hc3, language_formula_ols, df
)
glm_language = try_fit(
    "Reduced language-only fractional logit", fit_frac_logit, language_formula_glm, df
)
ols_text_full = try_fit(
    "Text-only full adjusted OLS-HC3", fit_ols_hc3, full_formula_ols, df_text
)
glm_text_full = try_fit(
    "Text-only full adjusted fractional logit",
    fit_frac_logit,
    full_formula_glm,
    df_text,
)
ols_text_context = try_fit(
    "Text-only reduced assignment-context OLS-HC3",
    fit_ols_hc3,
    context_formula_ols,
    df_text,
)
glm_text_context = try_fit(
    "Text-only reduced assignment-context fractional logit",
    fit_frac_logit,
    context_formula_glm,
    df_text,
)

fit_summary = pd.DataFrame([model_fit_row(m) for m in models])
print("\nModel fit summary")
display(fit_summary)
if model_errors:
    print("\nModel errors")
    display(pd.DataFrame(model_errors))

In [ ]:
# Tidy model outputs and prepare an APA-formatted coefficient table.
# - tidy_model: converts a single model object into a consistent tabular form.
# - If no models are present, produce an empty DataFrame to avoid downstream errors.
# - apa_table: formats estimates for reporting; estimate_digits controls numeric precision.
# Display a preview of the formatted table for inspection.
tidy_tables = [tidy_model(m) for m in models]
model_results = (
    pd.concat(tidy_tables, ignore_index=True) if tidy_tables else pd.DataFrame()
)
model_results_apa = apa_table(model_results, estimate_digits=3)
print("APA-ready coefficient table preview")
display(model_results_apa.head(80))

## 4. Estimated marginal means by assignment context

These are adjusted predictions from the **reduced assignment-context model**, with log word count held at the observed mean. This is intentionally preferable to a full model when language, major, university, year, and assignment type overlap too strongly.

In [ ]:
# =========================================================
# 9. Estimated marginal means by assignment context
# =========================================================
def estimated_marginal_means_context(res, data, model_label, outcome_scale="ols_100"):
    levels = list(pd.Index(data["assignment_context"].dropna().unique()).sort_values())
    grid = pd.DataFrame(
        {"assignment_context": levels, "log_word_count": data["log_word_count"].mean()}
    )
    pred = res.get_prediction(grid).summary_frame(alpha=0.05)
    out = grid.copy()
    out["model"] = model_label
    mean_col = "mean" if "mean" in pred.columns else pred.columns[0]
    ci_low_col = "mean_ci_lower" if "mean_ci_lower" in pred.columns else None
    ci_high_col = "mean_ci_upper" if "mean_ci_upper" in pred.columns else None
    factor = 100 if outcome_scale == "glm_prop" else 1
    out["estimated_mean_combined_ai_rate"] = pred[mean_col].to_numpy() * factor
    if ci_low_col and ci_high_col:
        out["CI_low"] = pred[ci_low_col].to_numpy() * factor
        out["CI_high"] = pred[ci_high_col].to_numpy() * factor
    out = out[
        [
            "model",
            "assignment_context",
            "log_word_count",
            "estimated_mean_combined_ai_rate",
            "CI_low",
            "CI_high",
        ]
    ]
    return out.sort_values("estimated_mean_combined_ai_rate", ascending=False)


emm_tables = []
if ols_context is not None:
    emm_tables.append(
        estimated_marginal_means_context(
            ols_context, df, "Reduced assignment-context OLS-HC3", "ols_100"
        )
    )
if glm_context is not None:
    emm_tables.append(
        estimated_marginal_means_context(
            glm_context, df, "Reduced assignment-context fractional logit", "glm_prop"
        )
    )
if ols_text_context is not None:
    emm_tables.append(
        estimated_marginal_means_context(
            ols_text_context,
            df_text,
            "Text-only reduced assignment-context OLS-HC3",
            "ols_100",
        )
    )
if glm_text_context is not None:
    emm_tables.append(
        estimated_marginal_means_context(
            glm_text_context,
            df_text,
            "Text-only reduced assignment-context fractional logit",
            "glm_prop",
        )
    )
emm_context = pd.concat(emm_tables, ignore_index=True) if emm_tables else pd.DataFrame()
print("Estimated marginal means by assignment context")
display(emm_context)

## 5. Term-level reduced-model comparisons

The next cell estimates approximate incremental fit changes by dropping one term at a time from the full OLS model. Because of the strong overlap among predictors, these values should be used only as sensitivity indicators, not as causal or definitive importance rankings.

In [ ]:
# =========================================================
# 10. Approximate term contribution via drop-one comparisons
# =========================================================
def drop_one_adj_r2(data, full_formula, terms_to_drop):
    rows = []
    full_res = smf.ols(full_formula, data=data).fit()
    full_adj = full_res.rsquared_adj
    for term_label, remove_pattern in terms_to_drop.items():
        reduced_formula = full_formula.replace(" + " + remove_pattern, "").replace(
            remove_pattern + " + ", ""
        )
        try:
            red_res = smf.ols(reduced_formula, data=data).fit()
            rows.append(
                {
                    "dropped_term": term_label,
                    "full_adjusted_R2": full_adj,
                    "reduced_adjusted_R2": red_res.rsquared_adj,
                    "delta_adjusted_R2": full_adj - red_res.rsquared_adj,
                    "reduced_formula": reduced_formula,
                }
            )
        except Exception as e:
            rows.append(
                {
                    "dropped_term": term_label,
                    "full_adjusted_R2": full_adj,
                    "reduced_adjusted_R2": np.nan,
                    "delta_adjusted_R2": np.nan,
                    "reduced_formula": reduced_formula,
                    "error": repr(e),
                }
            )
    return pd.DataFrame(rows).sort_values("delta_adjusted_R2", ascending=False)


terms_to_drop = {
    "language": "C(language)",
    "assignment_type": "C(assignment_type)",
    "major": "C(major)",
    "university": "C(university)",
    "year_grouping": "C(year_grouping)",
    "log_word_count": "log_word_count",
}
try:
    drop_one = drop_one_adj_r2(df, full_formula_ols, terms_to_drop)
except Exception as e:
    drop_one = pd.DataFrame({"error": [repr(e)]})
print("Drop-one adjusted R² comparisons")
display(drop_one)

## 6. Export APA-ready tables and manuscript paragraph

In [ ]:
# =========================================================
# 11. Export tables
# =========================================================
model_results_apa_export = model_results_apa.copy()
fit_summary_export = fit_summary.copy()
for c in ["adjusted_R2", "R2", "pseudo_R2_McFadden", "AIC", "BIC"]:
    if c in fit_summary_export.columns:
        fit_summary_export[c] = fit_summary_export[c].map(
            lambda x: "" if pd.isna(x) else round(float(x), 4)
        )

xlsx_path = OUTPUT_DIR / "Paper2_adjusted_multivariable_APA_tables.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    pd.DataFrame({"data_path": [str(DATA_PATH)], "n_rows": [len(df)]}).to_excel(
        writer, sheet_name="Data audit", index=False
    )
    category_counts.to_excel(writer, sheet_name="Category counts", index=False)
    assoc.to_excel(writer, sheet_name="Overlap diagnostics", index=False)
    (
        sparse_categories
        if len(sparse_categories)
        else pd.DataFrame({"message": ["No sparse categories below threshold."]})
    ).to_excel(writer, sheet_name="Sparse categories", index=False)
    vif_full.to_excel(writer, sheet_name="VIF full", index=False)
    vif_context.to_excel(writer, sheet_name="VIF context", index=False)
    fit_summary_export.to_excel(writer, sheet_name="Model fit", index=False)
    model_results_apa_export.to_excel(
        writer, sheet_name="Model coefficients", index=False
    )
    emm_context.to_excel(writer, sheet_name="EMM context", index=False)
    drop_one.to_excel(writer, sheet_name="Drop-one adj R2", index=False)
    if model_errors:
        pd.DataFrame(model_errors).to_excel(
            writer, sheet_name="Model errors", index=False
        )
    for name, ct in crosstab_dict.items():
        sheet = ("ct_" + name)[:31]
        ct.to_excel(writer, sheet_name=sheet)

category_counts.to_csv(OUTPUT_DIR / "category_counts.csv", index=False)
assoc.to_csv(OUTPUT_DIR / "categorical_overlap_diagnostics.csv", index=False)
vif_full.to_csv(OUTPUT_DIR / "vif_full_model.csv", index=False)
fit_summary_export.to_csv(OUTPUT_DIR / "model_fit_summary.csv", index=False)
model_results.to_csv(OUTPUT_DIR / "model_coefficients_raw.csv", index=False)
model_results_apa_export.to_csv(OUTPUT_DIR / "model_coefficients_APA.csv", index=False)
emm_context.to_csv(
    OUTPUT_DIR / "estimated_marginal_means_assignment_context.csv", index=False
)
drop_one.to_csv(OUTPUT_DIR / "drop_one_adjusted_R2.csv", index=False)
print("Exported workbook:", xlsx_path)
print("Exported output directory:", OUTPUT_DIR.resolve())

In [ ]:
# Manuscript-ready paragraph generator: produce a ready-to-use paragraph summarizing
# adjusted analyses and export to plain text and optionally DOCX.
def sig_terms(table, model_name, alpha=0.05):
    if table.empty:
        return []
    tmp = table[(table["model"] == model_name) & (table["p"] < alpha)].copy()
    tmp = tmp.sort_values("p")
    return tmp["term"].tolist()


def top_overlap_text(assoc_df, max_items=3):
    if assoc_df.empty:
        return ""
    top = assoc_df.sort_values("cramers_v", ascending=False).head(max_items)
    pieces = []
    for _, r in top.iterrows():
        pieces.append(
            f"{r['var1']} and {r['var2']} (Cramer's V = {r['cramers_v']:.2f})"
        )
    return "; ".join(pieces)


observed_context = (
    df.groupby("assignment_context")["combined_ai_rate_100"]
    .agg(n="count", mean="mean", sd="std")
    .sort_values("mean", ascending=False)
    .reset_index()
)
top_context = observed_context.iloc[0]
low_context = observed_context.iloc[-1]
overlap_sentence = top_overlap_text(assoc)
context_sig = sig_terms(model_results, "Reduced assignment-context OLS-HC3")
full_sig = sig_terms(model_results, "Full adjusted OLS-HC3")

paragraph = f"""
Adjusted analyses were conducted as sensitivity analyses because the corpus groupings were strongly overlapping. Crosstab and association diagnostics showed substantial dependency among grouping variables, especially {overlap_sentence}. Therefore, coefficients from the fully adjusted models should not be interpreted as cleanly separating independent effects of language, assignment type, major, university, or year grouping. In the reduced assignment-context model, which avoids the most serious aliasing by using non-overlapping assignment contexts and log word count, assignment context remained associated with the Combined AI rate. The highest observed context mean was for {top_context['assignment_context']} (M = {top_context['mean']:.2f}%, n = {int(top_context['n'])}), whereas the lowest was for {low_context['assignment_context']} (M = {low_context['mean']:.2f}%, n = {int(low_context['n'])}). Log word count was included in all models to account for file length. Across the full, reduced, fractional-logit, and text-only sensitivity models, results should be interpreted as adjusted corpus-level associations rather than causal effects or individual authorship judgments.
""".strip()
paragraph += "\n\nModel-screening note: significant coefficient terms in the full OLS model were "
paragraph += (
    (", ".join(full_sig[:12]) + ("..." if len(full_sig) > 12 else ""))
    if full_sig
    else "none at p < .05"
)
paragraph += (
    "; significant coefficient terms in the reduced assignment-context OLS model were "
)
paragraph += (
    (", ".join(context_sig[:12]) + ("..." if len(context_sig) > 12 else ""))
    if context_sig
    else "none at p < .05"
)
paragraph += "."

paragraph_path = OUTPUT_DIR / "manuscript_ready_adjusted_analysis_paragraph.txt"
paragraph_path.write_text(paragraph, encoding="utf-8")
print(paragraph)
print("\nSaved paragraph to:", paragraph_path)

if DOCX_AVAILABLE:
    doc = Document()
    doc.add_heading(
        "Paper 2 Adjusted Multivariable Analysis: Manuscript-Ready Output", level=1
    )
    # Brief interpretive qualifier included in the DOCX export
    doc.add_paragraph(
        "Note. These results should be interpreted as corpus-level associations, not causal effects or individual authorship judgments."
    )
    doc.add_heading("Manuscript-ready paragraph", level=2)
    doc.add_paragraph(paragraph)
    doc.add_heading("Model fit summary", level=2)
    table = doc.add_table(rows=1, cols=len(fit_summary_export.columns))
    for j, col in enumerate(fit_summary_export.columns):
        table.rows[0].cells[j].text = str(col)
    for _, row in fit_summary_export.iterrows():
        cells = table.add_row().cells
        for j, col in enumerate(fit_summary_export.columns):
            cells[j].text = str(row[col])
    doc.add_heading("Estimated marginal means by assignment context", level=2)
    if not emm_context.empty:
        emm_round = emm_context.copy()
        for c in [
            "log_word_count",
            "estimated_mean_combined_ai_rate",
            "CI_low",
            "CI_high",
        ]:
            if c in emm_round.columns:
                emm_round[c] = emm_round[c].map(
                    lambda x: "" if pd.isna(x) else f"{float(x):.2f}"
                )
        table = doc.add_table(rows=1, cols=len(emm_round.columns))
        for j, col in enumerate(emm_round.columns):
            table.rows[0].cells[j].text = str(col)
        for _, row in emm_round.iterrows():
            cells = table.add_row().cells
            for j, col in enumerate(emm_round.columns):
                cells[j].text = str(row[col])
    docx_path = OUTPUT_DIR / "Paper2_adjusted_multivariable_manuscript_output.docx"
    doc.save(docx_path)
    print("Saved DOCX:", docx_path)
else:
    print("python-docx is not available; skipped DOCX export.")

## Interpretation checklist for the manuscript

Use these rules when deciding what to report:

- Treat the **full adjusted model** as a sensitivity analysis if VIF and crosstabs show strong overlap.
- Give greater interpretive weight to **non-overlapping assignment-context models**.
- Report code-file analyses cautiously because code was scored with Pangram only.
- Do not claim causality.
- Do not infer individual student misconduct from detector scores.
- State clearly when predictors cannot be separated because of collinearity or near-perfect overlap.